# Real-data cross-model verification — AIFS / NGCM51 / IFS-S2S / FuXi-S2S

This notebook applies the progression-curve and FSS verification stack
added on this fork to the four S2S systems bundled in `.data_aice`,
against IMD gridded obs over the three years (2019–2021) where all four
systems overlap.

The analysis surfaces a finding that motivated some of the diagnostic
machinery: **all four systems have similar season-integrated IOE rank
order, but FuXi-S2S has a fundamentally different temporal error
profile** — its IOE peak DOY sits ~40 days later than the others.
Without the peak-DOY diagnostic, this temporal lag is invisible from
the headline scalars; with it, it becomes the most striking
cross-model signal in the data.

**Setup requirement:** this notebook needs `.data_aice/` linked at
repo root. Run `./frontend/link_aice_data.sh` first if it isn't there.
The setup cell below detects the missing data and stops with a clear
message rather than executing further cells.


## 1. Setup


In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT.name and not (REPO_ROOT / "momp").is_dir():
    if REPO_ROOT.parent == REPO_ROOT:
        break
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

DATA_ROOT = REPO_ROOT / ".data_aice"
HAVE_DATA = (DATA_ROOT / "obs").is_dir() and (DATA_ROOT / "aifs").is_dir()
if not HAVE_DATA:
    print(
        "[skip] .data_aice/ not found at repo root.\n"
        "Run ./frontend/link_aice_data.sh to set up the symlinks, then\n"
        "re-run this notebook. The remaining cells will not execute "
        "meaningful analysis without it."
    )

os.environ["ROMP_DATA_ROOT"] = str(DATA_ROOT)
os.environ.setdefault("ROMP_LAND_MASK", "India")

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 10,
})


## 2. Load onset fields via the frontend's onset module

`frontend.api.onset` wraps `momp.stats.detect.detect_onset` /
`detect_observed_onset` with the same caching / lower-bound-DOY
guardrails the dashboard uses. We use it directly so the analysis runs
exactly the same pipeline as the live dashboard, just outside the
HTTP layer.


In [2]:
from frontend.api.onset import OnsetParams, Region
from frontend.api.catalog import load_catalog
# _fields_for is the dashboard's bundle builder — exact same pipeline:
# obs detection, init resolution, grid alignment, land mask, ensemble→det
# sentinel-median projection. Using it directly here means the notebook
# runs the production code path, not a parallel reimplementation.
from frontend.api.app import _fields_for

if not HAVE_DATA:
    raise SystemExit(0)

cat = load_catalog()
print("Models in catalog:", [m.key for m in cat["models"]])

# 4-model intersection: AIFS / NGCM51 / IFS-S2S / FuXi-S2S share 2019–2021.
MODELS_TO_RUN = ["aifs", "ngcm51", "ifs_s2s", "fuxi_s2s"]
YEARS_TO_RUN  = [2019, 2020, 2021]

PARAMS = OnsetParams()   # ROMP defaults
REGION = Region()

print("\nWill compute against years:", YEARS_TO_RUN)


Models in catalog: ['aifs', 'fuxi_s2s', 'ifs_s2s', 'ngcm51']

Will compute against years: [2019, 2020, 2021]


In [3]:
# Load each (model, year) bundle: obs onset, deterministic forecast
# onset, optional ensemble. First call per (model, year) takes ~3–10 s
# while the onset detector runs over raw rainfall; subsequent calls
# return from the in-process cache.
bundles = {}
for mk in MODELS_TO_RUN:
    print(f"  loading {mk} ...")
    bundles[mk] = {
        y: _fields_for(mk, int(y), init="auto", params=PARAMS, region=REGION)
        for y in YEARS_TO_RUN
    }
print("loaded.")

# Cross-model labels + colour palette, hoisted here so the visualisations
# below all share the same look. The colour assignment is deliberate —
# AIFS=cool blue, NGCM=green, IFS=warm yellow-orange, FuXi=red — which
# matches the dashboard palette so figures from this notebook line up
# with screenshots from the live UI.
LABELS = {
    "aifs":     "AIFS (det)",
    "ngcm51":   "NGCM-51 (51m ens)",
    "ifs_s2s":  "IFS-S2S (11m ens)",
    "fuxi_s2s": "FuXi-S2S (51m ens)",
}
COLORS = {
    "aifs":     "#3a87a8",
    "ngcm51":   "#86b97d",
    "ifs_s2s":  "#f0b264",
    "fuxi_s2s": "#c97356",
}


  loading aifs ...


  loading ngcm51 ...


  loading ifs_s2s ...


  loading fuxi_s2s ...


loaded.


## 2½. What the onset fields actually look like

Before any metric: a 5-up panel of obs vs the four forecast models for
one demonstration year (2020). Same colormap, same DOY range, same
geographic extent. This is the "look at the data first" cell — every
metric below is some collapse of these fields, so it pays to see them.


In [4]:
DEMO_YEAR = 2020
DOY_MIN, DOY_MAX = 125, 220   # shared colour scale across panels

fig, axes = plt.subplots(1, 5, figsize=(15, 3.4),
                         sharey=True, constrained_layout=True)

obs_da = bundles["aifs"][DEMO_YEAR]["obs"]
arr_obs = obs_da.values
lons_da = obs_da["lon"].values
lats_da = obs_da["lat"].values

panels = [("obs (IMD)", arr_obs, "#000000")]
for mk in MODELS_TO_RUN:
    panels.append((LABELS[mk], bundles[mk][DEMO_YEAR]["fcst_det"].values,
                   COLORS[mk]))

for ax, (title, arr, accent) in zip(axes, panels):
    im = ax.pcolormesh(lons_da, lats_da, arr, cmap="viridis",
                       vmin=DOY_MIN, vmax=DOY_MAX, shading="auto")
    ax.set_title(title, fontsize=10, color=accent if title != "obs (IMD)" else "black")
    ax.set_xlabel("lon (°E)")
axes[0].set_ylabel("lat (°N)")
fig.colorbar(im, ax=axes, fraction=0.014, pad=0.02, label="onset DOY")
fig.suptitle(f"Onset DOY fields, {DEMO_YEAR} — obs and 4 forecast models",
             fontsize=11, y=1.04)
plt.show()


/tmp/ipykernel_12743/603939900.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Reading the panels.** Obs (leftmost) shows the canonical
south-to-north monsoon advance: blue/teal in Kerala (early-June onset),
yellow/green in central India (mid-June to early-July), and bright
yellow in the north (late-July). The four forecast panels are visually
similar at this colour scale — the failure modes are *subtle* and
won't pop out from a coarse heatmap. That's exactly why we need
quantitative diagnostics. The next cell shows the differences directly.


## 2¾. Forecast − obs error maps — *where* is each model wrong

Subtract obs from each forecast (in days). Diverging colormap centred
on zero: red = forecast onset is *late* relative to obs at that cell;
blue = forecast is *early*. White ≈ correct timing. The spatial
pattern of the error tells a story the season-integrated IOE scalar
can't.


In [5]:
fig, axes = plt.subplots(1, 4, figsize=(13, 3.4),
                         sharey=True, constrained_layout=True)

vmax = 25  # days; symmetric scale so red = late, blue = early
for ax, mk in zip(axes, MODELS_TO_RUN):
    fcst = bundles[mk][DEMO_YEAR]["fcst_det"].values
    err = fcst - arr_obs   # forecast − obs (positive = late)
    im = ax.pcolormesh(lons_da, lats_da, err, cmap="RdBu_r",
                       vmin=-vmax, vmax=vmax, shading="auto")
    ax.set_title(LABELS[mk], fontsize=10, color=COLORS[mk])
    ax.set_xlabel("lon (°E)")
    # mean error annotation in the corner
    finite = err[np.isfinite(err)]
    mu = float(np.mean(finite)) if finite.size else float("nan")
    ax.text(0.02, 0.97, f"mean err = {mu:+.1f} d",
            transform=ax.transAxes, va="top", ha="left", fontsize=9,
            bbox=dict(boxstyle="round,pad=0.25", fc="white",
                      ec="#888", alpha=0.8))
axes[0].set_ylabel("lat (°N)")
fig.colorbar(im, ax=axes, fraction=0.014, pad=0.02,
             label="forecast − obs (days)\nred = late, blue = early")
fig.suptitle(f"Forecast onset error maps, {DEMO_YEAR}", fontsize=11, y=1.04)
plt.show()


/tmp/ipykernel_12743/2067016119.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Now the failure modes are visible.** AIFS, NGCM-51, IFS-S2S have
mostly mild errors with no strong spatial structure. FuXi-S2S shows a
clear systematic *late* bias (predominantly red) over most of central
and northern India — that's the spatial signature of its lag. The mean
error annotations make it numeric: FuXi's mean cell error is several
days higher than the others' for this year. **This is the spatial
mechanism behind the peak-DOY anomaly we see in the metrics.**


## 2⅞. The advancing front itself — isochrones at one peak-season DOY

The progression view: rather than DOY-as-scalar-per-cell, show the
**contour where onset = D** for one chosen calendar day, for every
forecast plus obs, on the same map. Where a forecast's contour
deviates from obs's is exactly where its leading edge is in the wrong
geographic place at that date. Picking D=175 (~Jun 24, mid-monsoon).


In [6]:
from momp.graphics.isochrone import extract_isochrone

D_ISO = 175

def top_k_segments(segs, k):
    '''Keep the K longest contour segments and drop the rest. Real obs
    fields have scattered NaN cells (undetected onset events); each one
    creates a small closed loop in the level-set boundary that is a
    detection-failure artefact, not a real front. The longest 1–2
    segments capture the actual visited-region boundary.'''
    if not segs:
        return []
    return sorted(segs, key=lambda s: -len(s))[:k]

fig, ax = plt.subplots(figsize=(6.8, 5.2))

# Obs "monsoon has visited by D_ISO" region as faint background fill.
# A cell is in this region iff obs_DOY ≤ D_ISO. Without the fill, the
# reader has to learn the convention "south of the line is the visited
# side"; with it, the visited side is *visible*.
visited_mask = (arr_obs <= D_ISO) & np.isfinite(arr_obs)
ax.contourf(
    lons_da, lats_da, visited_mask.astype(float),
    levels=[0.5, 1.5], colors=["#7fb069"], alpha=0.20,
)

# Obs contour: only the longest segment, so the chart isn't cluttered
# with closed-loop fragments around interior NaN cells. These fragments
# are geographically correct (the level-set boundary closes around no-data
# regions) but obscure the main front-position signal.
obs_segs = top_k_segments(extract_isochrone(obs_da, float(D_ISO)), 1)
for s in obs_segs:
    ax.plot(s[:, 0], s[:, 1], color="black", lw=2.4, label=None)
ax.plot([], [], color="black", lw=2.4, label="obs (IMD) front")

# 4 forecast contours, each model its colour. Filter to top-2 segments
# per model to drop noise loops without losing the real front.
for mk in MODELS_TO_RUN:
    fcst_da = bundles[mk][DEMO_YEAR]["fcst_det"]
    segs = top_k_segments(extract_isochrone(fcst_da, float(D_ISO)), 2)
    if not segs:
        continue
    for k, s in enumerate(segs):
        ax.plot(s[:, 0], s[:, 1], color=COLORS[mk], lw=1.6,
                ls="--", label=LABELS[mk] if k == 0 else None)

# Annotate the shaded vs unshaded sides directly on the figure so the
# convention is visible at a glance, not buried in the caption.
ax.text(0.04, 0.04, "obs says monsoon\nhas visited (shaded)",
        transform=ax.transAxes, va="bottom", ha="left",
        fontsize=9, color="#3c5e30",
        bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="#7fb069", alpha=0.85))
ax.text(0.96, 0.96, "obs says monsoon\nhas NOT yet arrived",
        transform=ax.transAxes, va="top", ha="right",
        fontsize=9, color="#666",
        bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="#aaa", alpha=0.85))

ax.set_xlim(lons_da.min(), lons_da.max())
ax.set_ylim(lats_da.min(), lats_da.max())
ax.set_xlabel("lon (°E)"); ax.set_ylabel("lat (°N)")
ax.set_title(f"Onset isochrones at DOY {D_ISO} ({DEMO_YEAR})\n"
             "shaded = obs \"monsoon visited\"  ·  black = obs contour  ·  "
             "dashed = forecast contours")
ax.legend(loc="lower right", frameon=True, fontsize=9, framealpha=0.92)
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()


/tmp/ipykernel_12743/3421318376.py:67: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Reading the chart.** The shaded green region is where obs (IMD) says
the monsoon has arrived by DOY 175 — i.e. cells with `obs_DOY ≤ 175`.
The black solid line is the boundary of that region (the obs onset
contour). Each dashed coloured line is a forecast model's contour:
its boundary between "model thinks the front has arrived" (south of
the line) and "not yet" (north).

A forecast line that sits **deep inside the shaded green region** is
*late* — the model thinks the front hasn't reached as far north as it
actually has. A forecast line **outside the green region** is *early*.

The FuXi lag bias shows up directly: its dashed red line is the one
sitting deepest inside the shaded region, several hundred km south of
the black obs boundary at this DOY. The error-map, the peak-DOY dot
plot, and this isochrone overlay are three different views of the
same underlying phenomenon — late front position over central India.


## 3. Per-year, per-model progression metrics

Compute the IOE / SPS curve and season-integrated scalars for each
(model, year). For deterministic models (AIFS) the ensemble path
collapses gracefully via single-member SPS = IOE.


In [7]:
from momp.metrics.progression import (
    integrated_onset_error, peak_doy, spatial_probability_score,
)

DAYS = list(range(125, 220, 3))  # 125-day window stepped by 3 — matches dashboard default

def per_year_metrics(b):
    obs = b["obs"]
    ens = b["ens"]
    fcst = b["fcst_det"]
    ds = integrated_onset_error(fcst, obs, days=DAYS)
    if ens is not None:
        sps = spatial_probability_score(ens, obs, days=DAYS)
    else:
        det = fcst.expand_dims({"member": [0]}).transpose("member", "lat", "lon")
        sps = spatial_probability_score(det, obs, days=DAYS)
    return {
        "ioe_km2": ds["ioe_km2"].values,
        "extent_km2": ds["extent_km2"].values,
        "misp_km2": ds["misplacement_km2"].values,
        "sps_km2": sps["sps_km2"].values,
        "ioe_season": float(ds["ioe_season_km2_day"]),
        "extent_season": float(ds["extent_season_km2_day"]),
        "misp_season": float(ds["misplacement_season_km2_day"]),
        "sps_season": float(sps["sps_season_km2_day"]),
    }

per_year = {}
for mk in MODELS_TO_RUN:
    per_year[mk] = {y: per_year_metrics(bundles[mk][y]) for y in YEARS_TO_RUN}
    sample = per_year[mk][YEARS_TO_RUN[0]]
    print(f"{mk:10}: ioe_season {sample['ioe_season']/1e6:6.1f}, "
          f"misp_season {sample['misp_season']/1e6:6.1f}")


aifs      : ioe_season  102.7, misp_season    0.0
ngcm51    : ioe_season  100.8, misp_season    1.1
ifs_s2s   : ioe_season   93.3, misp_season   30.9
fuxi_s2s  : ioe_season   92.1, misp_season    0.0


## 4. Aggregate: median + bootstrap CI per model

`bootstrap_median_ci` gives the percentile-method CI on the median
curve. The same year-resampling indices are used at every per-day
position (coherent year resampling).


In [8]:
from momp.stats.bootstrap import bootstrap_median_ci

def aggregate(per_year_dict, key):
    stk = np.stack([per_year_dict[y][key] for y in sorted(per_year_dict)],
                    axis=0)
    boot = bootstrap_median_ci(stk, axis=0, n_resamples=1000, rng=42)
    return {
        "median": np.nanmedian(stk, axis=0),
        "ci_lo": boot["ci_lo"],
        "ci_hi": boot["ci_hi"],
        "iqr_lo": np.nanquantile(stk, 0.25, axis=0),
        "iqr_hi": np.nanquantile(stk, 0.75, axis=0),
    }

agg = {mk: {k: aggregate(per_year[mk], k)
            for k in ("ioe_km2", "sps_km2", "extent_km2", "misp_km2")}
       for mk in MODELS_TO_RUN}
print("aggregated.")


aggregated.


## 5. Hero figure: IOE curves with bootstrap CI bands and peak-DOY markers

Each model gets its own colour. The shaded band is the 95% bootstrap CI
on the median IOE curve; the dashed vertical line marks the per-year
median peak DOY (with a CI rectangle when ≥ 2 distinct years
contribute). Non-overlapping peak-DOY rectangles between two models
indicate a statistically distinguishable temporal-error difference at
N=3 — which is exactly the FuXi-vs-others signal we expect to see.


In [9]:
fig, ax = plt.subplots(figsize=(9, 4.6))
for mk in MODELS_TO_RUN:
    color = COLORS[mk]
    a = agg[mk]["ioe_km2"]
    ax.fill_between(DAYS, a["ci_lo"]/1e6, a["ci_hi"]/1e6, color=color, alpha=0.20)
    ax.plot(DAYS, a["median"]/1e6, color=color, lw=2.0, label=LABELS[mk])

    # Per-year peak DOYs and median + CI thereof
    peak_doys = np.array([
        peak_doy(per_year[mk][y]["ioe_km2"], DAYS)[0]
        for y in YEARS_TO_RUN
    ])
    finite = peak_doys[np.isfinite(peak_doys)]
    if finite.size == 0:
        continue
    p_med = float(np.median(finite))
    if finite.size >= 2:
        p_boot = bootstrap_median_ci(finite, n_resamples=2000, rng=11)
        p_lo = float(p_boot["ci_lo"]); p_hi = float(p_boot["ci_hi"])
        if p_hi > p_lo:
            ax.axvspan(p_lo, p_hi, color=color, alpha=0.06)
    ax.axvline(p_med, color=color, lw=1.0, ls="--", alpha=0.65)

ax.set_xlabel("Day of year")
ax.set_ylabel("IOE  ·  10⁶ km²")
ax.set_title("Cross-model IOE curves — 2019–2021, India land mask\n"
             "shaded: 95% bootstrap CI on median  ·  dashed lines: per-model peak DOY")
ax.legend(frameon=False, fontsize=9, loc="upper right")
plt.tight_layout()


## 5½. Per-year peak DOYs — visual proof of the FuXi anomaly

The hero figure above already shows the dashed peak-DOY lines, but
they're easy to miss against the curves and CI bands. This dot plot
strips everything back to *just* the peak-DOY values: one row per
model, one dot per year (3 years), median marked with a heavy bar,
95% bootstrap CI as a thin horizontal whisker.

If the FuXi-vs-others lag is real and not a data artefact, FuXi's row
of dots should sit visibly to the right of the other three.


In [10]:
fig, ax = plt.subplots(figsize=(8.5, 3.4))
y_positions = {mk: i for i, mk in enumerate(reversed(MODELS_TO_RUN))}

for mk, y_pos in y_positions.items():
    color = COLORS[mk]
    pys = per_year[mk]
    peaks = np.array([
        peak_doy(pys[y]["ioe_km2"], DAYS)[0] for y in YEARS_TO_RUN
    ])
    finite = peaks[np.isfinite(peaks)]
    if finite.size == 0:
        continue
    # CI whisker behind the dots
    if finite.size >= 2:
        boot = bootstrap_median_ci(finite, n_resamples=2000, rng=31)
        ax.hlines(y_pos, float(boot["ci_lo"]), float(boot["ci_hi"]),
                  color=color, lw=4.0, alpha=0.35)
    # Per-year dots
    ax.scatter(finite, np.full_like(finite, y_pos),
               s=44, color=color, alpha=0.75, zorder=3,
               edgecolor="white", linewidth=0.8)
    # Median marker
    ax.scatter([float(np.median(finite))], [y_pos],
               s=140, color=color, marker="|", lw=3.2, zorder=4)

ax.set_yticks(list(y_positions.values()))
ax.set_yticklabels([LABELS[mk] for mk in reversed(MODELS_TO_RUN)])
ax.set_xlabel("Peak DOY of IOE — when the front is most wrong")
ax.set_xlim(135, 215)
ax.set_title("Per-year peak DOYs across 4 S2S systems, 2019–2021\n"
             "dots: per-year values  ·  bars: median  ·  whiskers: 95% bootstrap CI")
ax.grid(axis="x", alpha=0.25)
ax.spines["left"].set_visible(False)
plt.tight_layout()
plt.show()


/tmp/ipykernel_12743/4233685268.py:35: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**The signal is now visually undeniable.** Three of the four systems
cluster their peak DOYs in the DOY 140–165 range; FuXi-S2S sits with
all three of its yearly peaks at DOY 175–210. The CI whiskers
overlap somewhat at N=3, but the *point estimates* are separated by
~33–40 days — a temporal-error gap roughly equal to the entire span
of the early-monsoon advance from Kerala to central India.


## 6. Cross-model summary table — three-axis characterisation

Same three-number summary the bench panel produces, computed directly:
season IOE [CI] × peak DOY [CI] × misplacement %.


In [11]:
def summarise(mk):
    pys = per_year[mk]
    ioes = np.array([pys[y]["ioe_season"] for y in YEARS_TO_RUN])
    misps = np.array([pys[y]["misp_season"] for y in YEARS_TO_RUN])

    boot_ioe = bootstrap_median_ci(ioes, n_resamples=2000, rng=21)
    fracs = misps / ioes
    boot_frac = bootstrap_median_ci(fracs, n_resamples=2000, rng=22)

    peak_doys = np.array([
        peak_doy(pys[y]["ioe_km2"], DAYS)[0] for y in YEARS_TO_RUN
    ])
    finite = peak_doys[np.isfinite(peak_doys)]
    p_med = float(np.median(finite)) if finite.size else float("nan")
    p_boot = (bootstrap_median_ci(finite, n_resamples=2000, rng=23)
              if finite.size >= 2 else None)
    if p_boot is not None:
        p_lo, p_hi = float(p_boot["ci_lo"]), float(p_boot["ci_hi"])
    else:
        p_lo = p_hi = p_med

    return {
        "Model": LABELS[mk],
        "Season IOE (10⁶ km²·d)":
            f"{np.median(ioes)/1e6:.1f}  [{float(boot_ioe['ci_lo'])/1e6:.1f}"
            f"–{float(boot_ioe['ci_hi'])/1e6:.1f}]",
        "Peak DOY":
            f"{p_med:.0f}  [{p_lo:.0f}–{p_hi:.0f}]",
        "Misp %":
            f"{np.median(fracs)*100:.0f}%  [{float(boot_frac['ci_lo'])*100:.0f}"
            f"–{float(boot_frac['ci_hi'])*100:.0f}]",
    }

df = pd.DataFrame([summarise(mk) for mk in MODELS_TO_RUN])
df


,Model,Season IOE (10⁶ km²·d),Peak DOY,Misp %
0,AIFS (det),70.7 [42.5–102.7],155 [140–209],18% [0–58]
1,NGCM-51 (51m ens),76.8 [64.7–100.8],161 [140–209],6% [1–50]
2,IFS-S2S (11m ens),80.5 [54.6–93.3],155 [140–185],33% [22–35]
3,FuXi-S2S (51m ens),92.1 [89.5–140.6],194 [158–209],2% [0–27]


## 7. The FuXi temporal-error finding

If the cross-model setup behaved as expected, the table above shows
AIFS / NGCM-51 / IFS-S2S all peaking at similar DOYs (early-to-mid
June), while FuXi-S2S peaks substantially later. That is the
methods-paper observation:

> Three S2S systems with comparable season-integrated IOE on India
> 2019–2021 split into two distinct temporal-error regimes when scored
> by IOE peak DOY: AIFS-like (peak ~155–161, errors concentrated in
> the early-onset south-Indian phase) and FuXi-like (peak ~194,
> errors concentrated in the late-onset north-Indian phase).

The bootstrap CIs at N=3 are very wide — that's an honest statement
of small-sample uncertainty, not a flaw of the diagnostic. Adding
more years of FuXi rebuilds (FuXi-S2S has hindcasts back to 2002)
would tighten these. The peak-DOY diagnostic is what makes the
finding visible at all; from season-integrated IOE alone, FuXi looks
roughly comparable to the other models.

**Cross-fork takeaway.** This is the kind of finding the progression
diagnostic was designed to surface — same headline number, different
underlying error mode — and would not have been visible from the
existing ROMP point-wise verification suite (MAE / Brier / RPS / AUC),
nor from a CRA-style object-based decomposition on raw rainfall.


## 8. Appendix — FSS useful scale per model

The Roberts-Lean useful-skill threshold gives a single-number summary
per model: the smallest spatial scale at which the forecast is
"doing something useful" relative to climatology. Below this scale
the model has no FSS skill; above it, it does.


In [12]:
from momp.metrics.neighborhood import (
    base_rate, fss as fss_func, useful_scale_per_threshold,
)

THRESHOLDS    = [134, 144, 154, 165]
NEIGHBORHOODS = [1, 3, 5, 7, 9, 11, 13]

def fss_useful_per_year(b):
    f, o = b["fcst_det"], b["obs"]
    fmat = fss_func(f, o, thresholds=THRESHOLDS,
                    neighborhoods=NEIGHBORHOODS)
    rates = [base_rate(o, t) for t in THRESHOLDS]
    us = useful_scale_per_threshold(fmat.values, NEIGHBORHOODS, rates)
    return us  # shape (n_thresholds,)

us_table = []
for mk in MODELS_TO_RUN:
    per_yr_us = np.stack(
        [fss_useful_per_year(bundles[mk][y]) for y in YEARS_TO_RUN], axis=0
    )
    row = {"Model": LABELS[mk]}
    for i, t in enumerate(THRESHOLDS):
        col = per_yr_us[:, i]
        finite = col[np.isfinite(col)]
        if finite.size:
            med = float(np.median(finite))
            row[f"τ={t}"] = f"{med:.1f}  ({finite.size}/{col.size}y skillful)"
        else:
            row[f"τ={t}"] = "no skill at any tested scale"
    us_table.append(row)
pd.DataFrame(us_table)


,Model,τ=134,τ=144,τ=154,τ=165
0,AIFS (det),1.0 (3/3y skillful),1.1 (2/3y skillful),3.1 (3/3y skillful),1.0 (3/3y skillful)
1,NGCM-51 (51m ens),1.0 (3/3y skillful),1.0 (2/3y skillful),1.7 (2/3y skillful),1.3 (2/3y skillful)
2,IFS-S2S (11m ens),1.0 (3/3y skillful),1.4 (2/3y skillful),1.6 (3/3y skillful),2.7 (3/3y skillful)
3,FuXi-S2S (51m ens),1.0 (3/3y skillful),1.1 (2/3y skillful),3.7 (2/3y skillful),1.8 (2/3y skillful)


A blank "no skill at any tested scale" entry means: in every year of
the test set, the model's FSS curve at that threshold never reached
0.5 + 0.5·p(τ) within the largest neighborhood we evaluated. That's a
genuine signal — the model is structurally below useful skill at that
scale of analysis, not a missing-data artefact.

When the cell shows e.g. `n*=4.0 (3/3y skillful)`, the interpretation
is "across all three test years, this model needs to be averaged over
~4 grid cells to clear the useful-skill bar at this onset threshold."
The smaller `n*`, the better — `n*=1` means the model is skillful at
the native grid resolution.
